In [ ]:
import requests
import time
import json
from datetime import datetime, timedelta, timezone



# Constants
INTERVAL_MINUTES = 15
DAYS_BACK = 1  # ~2 months
MARKET_SLUG_PREFIX = "btc-updown-15m-"
API_URL = "https://gamma-api.polymarket.com/markets/slug/"

# Calculate the current and start timestamps (UTC)

# Use timezone-aware UTC datetime
now = datetime.now(timezone.utc)
end_time = now.replace(minute=0, second=0, microsecond=0) + timedelta(minutes=INTERVAL_MINUTES)
start_time = end_time - timedelta(days=DAYS_BACK)

# Generate all 15-min interval timestamps between start and end
intervals = []
t = start_time
while t < end_time:
    # Polymarket slugs use UNIX timestamp in seconds
    intervals.append(int(t.timestamp()))
    t += timedelta(minutes=INTERVAL_MINUTES)

# Fetch data for each interval
all_data = []
total = len(intervals)
for idx, ts in enumerate(intervals, 1):
    slug = f"{MARKET_SLUG_PREFIX}{ts}"
    url = f"{API_URL}{slug}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            # Only append if the market exists and is for BTC 15m
            if data and data.get("slug", "").startswith("btc-updown-15m-"):
                all_data.append(data)
        else:
            # Market may not exist for every interval
            pass
    except Exception as e:
        print(f"Error fetching {url}: {e}")
    if idx % 10 == 0 or idx == total:
        print(f"Processed {idx}/{total} intervals...")
    time.sleep(0.1)  # Be polite to the API

# Save to JSON file
with open("polymarket_btc15m_all.json", "w") as f:
    json.dump(all_data, f, indent=2)

print(f"Saved {len(all_data)} markets to polymarket_btc15m_all.json")


Alternatively get by series

In [ ]:
import requests
import json

url = "https://gamma-api.polymarket.com/series/10192?include_chat=false"

response = requests.get(url)

with open("polymarket_series_10192.json", "w") as f:
    json.dump(response.json(), f, indent=2)

print(f"Saved {len(response.content)} bytes to polymarket_series_10192.json")